In [2]:
import pandas as pd
import datetime
import os

In [4]:
print(os.getcwd())

/Users/harrybrowne/Downloads/Imperial Work/SystematicTrading/stml


In [5]:
data = pd.read_excel("DATA_SYS_PASTED.xlsx")

In [6]:
data.head()

,Unnamed: 0,VIX,Unnamed: 2,VIX3M,Unnamed: 4,MOVE,Unnamed: 6,DXY,Unnamed: 8,CBOE_SKEW,...,Unnamed: 34,US_ISM_MFG_PMI,Unnamed: 36,CHINA_PMI_MFG,Unnamed: 38,GERMANY_PMI_MFG,Unnamed: 40,TIPS10Y,Unnamed: 42,BE10Y
0,1990-01-02,17.24,1990-01-02,NaN,1990-01-02,91.73,1990-01-02,94.25,1990-01-02,126.09,...,1990-01-02,47.4,1990-01-02,NaN,1990-01-02,NaN,1990-01-02,NaN,1990-01-02,NaN
1,1990-01-03,18.19,1990-01-03,NaN,1990-01-03,93.21,1990-01-03,94.45,1990-01-03,123.34,...,1990-01-03,47.4,1990-01-03,NaN,1990-01-03,NaN,1990-01-03,NaN,1990-01-03,NaN
2,1990-01-04,19.22,1990-01-04,NaN,1990-01-04,95.09,1990-01-04,92.54,1990-01-04,122.62,...,1990-01-04,47.4,1990-01-04,NaN,1990-01-04,NaN,1990-01-04,NaN,1990-01-04,NaN
3,1990-01-05,20.11,1990-01-05,NaN,1990-01-05,85.28,1990-01-05,93.06,1990-01-05,121.27,...,1990-01-05,47.4,1990-01-05,NaN,1990-01-05,NaN,1990-01-05,NaN,1990-01-05,NaN
4,1990-01-08,20.26,1990-01-08,NaN,1990-01-06,85.28,1990-01-08,92.05,1990-01-08,124.12,...,1990-01-06,47.4,1990-01-06,NaN,1990-01-06,NaN,1990-01-08,NaN,1990-01-08,NaN


In [44]:
unnamed_columns = [f'Unnamed: {i}' for i in range(0,43,2)]

actual_columns = data.columns.difference(unnamed_columns)

print(actual_columns)

normal_date_columns = actual_columns.difference([ 'MOVE',
    'EIA_CRUDE_STOCK', 'EIA_DIST_STOCK', 'EIA_GASOLINE_STOCK',
    'EIA_NG_STOCK', 'CHINA_PMI_MFG', 'GERMANY_PMI_MFG', 'US_ISM_MFG_PMI'
])

Index(['10Y_BUND', '10Y_UST', '2Y_UST', 'BAL_DRY_INDEX', 'BE10Y', 'CBOE_SKEW',
       'CHINA_PMI_MFG', 'DXY', 'EIA_CRUDE_STOCK', 'EIA_DIST_STOCK',
       'EIA_GASOLINE_STOCK', 'EIA_NG_STOCK', 'EURUSD', 'GERMANY_PMI_MFG',
       'HY_OAS', 'IG_OAS', 'LME_COPPER_STOCK', 'MOVE', 'TIPS10Y',
       'US_ISM_MFG_PMI', 'VIX', 'VIX3M'],
      dtype='object')


In [65]:
cols_to_keep = [c for col in normal_date_columns 
                for c in [data.columns[data.columns.get_loc(col) - 1], col]
                if data.columns.get_loc(col) > 0]

df = data[cols_to_keep]


In [66]:
for i in [14, 12, 10, 30, 42, 8, 6, 28, 24, 26, 32, 40, 0, 2]:
    df[f'Unnamed: {i}'] = pd.to_datetime(df[f'Unnamed: {i}'])

/var/folders/hq/6kqqgzcx25j9dc7vcs2_ghz00000gn/T/ipykernel_13221/2445674318.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'Unnamed: {i}'] = pd.to_datetime(df[f'Unnamed: {i}'])
/var/folders/hq/6kqqgzcx25j9dc7vcs2_ghz00000gn/T/ipykernel_13221/2445674318.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'Unnamed: {i}'] = pd.to_datetime(df[f'Unnamed: {i}'])
/var/folders/hq/6kqqgzcx25j9dc7vcs2_ghz00000gn/T/ipykernel_13221/2445674318.py:2: SettingWithCopyWarning: 
A value is trying to be set on a 

In [67]:
unnamed_date_cols = [f'Unnamed: {i}' for i in [14, 12, 10, 30, 42, 8, 6, 28, 24, 26, 32, 40, 0, 2]]

reference = data[unnamed_date_cols[0]].dropna().values  # use raw values to compare order explicitly

all_same = all(
    (data[col].dropna().values == reference).all()
    for col in unnamed_date_cols[1:]
)

print(all_same)



True


In [ ]:
df = df.set_index(df['Unnamed: 14'])
df.index.name = 'Date'

df = df.drop(columns= unnamed_date_cols)

df = df.dropna(subset=['10Y_BUND'])

In [73]:
other_cols = [ 'MOVE',
    'EIA_CRUDE_STOCK', 'EIA_DIST_STOCK', 'EIA_GASOLINE_STOCK',
    'EIA_NG_STOCK', 'CHINA_PMI_MFG', 'GERMANY_PMI_MFG', 'US_ISM_MFG_PMI'
]


all_cols = data.columns.tolist()

def get_left_col_df(data, col):
    idx = all_cols.index(col)
    left_col = all_cols[idx - 1]
    
    # Extract the pair
    temp = data[[left_col, col]].copy()
    
    # Convert left col to datetime (handles both datetime and Excel serial numbers)
    if not pd.api.types.is_datetime64_any_dtype(temp[left_col]):
        try:
            # Try Excel serial number conversion first
            temp[left_col] = pd.to_datetime(temp[left_col], unit='D', origin='1899-12-30')
        except:
            temp[left_col] = pd.to_datetime(temp[left_col])
    
    # Drop rows where date is not in df's index
    temp = temp[temp[left_col].isin(df.index)]
    
    # Set the date column as index for joining
    temp = temp.set_index(left_col)
    temp.index.name = df.index.name
    
    return temp[[col]]

# Build and join each sub-dataframe onto df
for col in other_cols:
    temp = get_left_col_df(data, col)
    df = df.join(temp, how='left')

print(df.shape)
print(df[other_cols].head())

(8478, 22)
             MOVE  EIA_CRUDE_STOCK  EIA_DIST_STOCK  EIA_GASOLINE_STOCK  \
Date                                                                     
1990-01-02  91.73           324280          106744                 NaN   
1990-01-03  93.21           324280          106744                 NaN   
1990-01-04  95.09           324280          106744                 NaN   
1990-01-05  85.28           324222          109207            210982.0   
1990-01-08  86.91           324222          109207            210982.0   

            EIA_NG_STOCK  CHINA_PMI_MFG  GERMANY_PMI_MFG  US_ISM_MFG_PMI  
Date                                                                      
1990-01-02           NaN            NaN              NaN            47.4  
1990-01-03           NaN            NaN              NaN            47.4  
1990-01-04           NaN            NaN              NaN            47.4  
1990-01-05           NaN            NaN              NaN            47.4  
1990-01-08          

In [91]:
df = df.drop(columns=['GERMANY_PMI_MFG'])
df.to_csv('alternate_data_cleaned.csv')

In [ ]:
ohlcv = pd.read_csv('data/ohlcv_data.csv')



In [92]:
df.tail()

,10Y_BUND,10Y_UST,2Y_UST,BAL_DRY_INDEX,BE10Y,CBOE_SKEW,DXY,EURUSD,HY_OAS,IG_OAS,...,TIPS10Y,VIX,VIX3M,MOVE,EIA_CRUDE_STOCK,EIA_DIST_STOCK,EIA_GASOLINE_STOCK,EIA_NG_STOCK,CHINA_PMI_MFG,US_ISM_MFG_PMI
Date,,,,,,,,,,,,,,,,,,,,,
2022-06-24,1.442,3.1301,3.0632,2331.0,2.5744,120.43,104.185,1.0553,5.07,1.48,...,0.5557,27.23,28.95,127.00,415566,112401,221608.0,2251.0,49.6,55.9
2022-06-27,1.547,3.1997,3.1212,2295.0,2.5444,119.98,103.939,1.0584,5.02,1.48,...,0.6517,26.95,28.18,133.22,415566,112401,221608.0,2251.0,49.6,55.9
2022-06-28,1.628,3.1715,3.1096,2204.0,2.4741,120.46,104.506,1.0519,5.21,1.51,...,0.6945,28.36,29.15,135.22,415566,112401,221608.0,2251.0,49.6,55.9
2022-06-29,1.519,3.0891,3.0385,2186.0,2.3917,119.26,105.106,1.0442,5.49,1.53,...,0.6937,28.16,29.00,130.43,415566,112401,221608.0,2251.0,49.6,55.9
2022-06-30,1.336,3.0129,2.9533,2240.0,2.3449,119.94,104.685,1.0484,5.69,1.55,...,0.6626,28.71,29.71,135.50,415566,112401,221608.0,2251.0,50.2,53.5
